# Experiment 4: Context Truncation Analysis

Compute the distribution of code sequence token lengths to quantify what fraction of the corpus exceeds standard LLM context windows (2048, 4096, 8192 tokens). This validates the theoretical critique about signal loss via truncation.

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd

from utils.data_loader import load_partition, print_mode_banner, is_validation_mode
from utils.features import tokenize_with_tiktoken
from utils.plotting import setup_style, plot_cdf, COLORS
import matplotlib.pyplot as plt
import seaborn as sns

VALIDATION_MODE = is_validation_mode()
print_mode_banner(VALIDATION_MODE)
setup_style()

## 1. Load All Partitions

In [ ]:
partitions = {}
for space in ['APPS', 'CDSS', 'KBSS']:
    df = load_partition(space, validation_mode=VALIDATION_MODE)
    partitions[space] = df
    print(f"{space}: {len(df):,} rows")

## 2. Tokenize with tiktoken (cl100k_base)
Using OpenAI's cl100k_base BPE encoding to simulate transformer token boundaries. This is the encoding used by GPT-4 and similar models.

In [ ]:
token_lengths = {}
for space, df in partitions.items():
    print(f"\nTokenizing {space} ({len(df):,} rows)...")
    lengths = tokenize_with_tiktoken(df['input'], encoding_name='cl100k_base')
    token_lengths[space] = lengths
    
    print(f"  Token length statistics:")
    print(f"    Mean:   {lengths.mean():.1f}")
    print(f"    Median: {np.median(lengths):.1f}")
    print(f"    Min:    {lengths.min()}")
    print(f"    Max:    {lengths.max()}")
    print(f"    Std:    {lengths.std():.1f}")
    print(f"    P95:    {np.percentile(lengths, 95):.0f}")
    print(f"    P99:    {np.percentile(lengths, 99):.0f}")

## 3. Token Length Distribution
Histograms showing the distribution of token lengths per partition.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Token Length Distributions (cl100k_base BPE)', 
             fontsize=16, fontweight='bold', color=COLORS['highlight'])

for ax, (space, lengths) in zip(axes, token_lengths.items()):
    ax.hist(lengths, bins=50, color=COLORS[space], alpha=0.7, edgecolor='black')
    ax.axvline(2048, color=COLORS['warning'], linestyle='--', linewidth=2, label='2048')
    ax.axvline(4096, color=COLORS['primary'], linestyle='--', linewidth=2, label='4096')
    ax.set_title(f'{space}', fontweight='bold')
    ax.set_xlabel('Token Length')
    ax.set_ylabel('Count')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.2)

plt.tight_layout()
plt.show()

## 4. Cumulative Distribution Function (CDF) with Threshold Markers
The CDF shows what fraction of codes fit within each context window.

In [ ]:
thresholds = [2048, 4096, 8192]

fig = plot_cdf(
    token_lengths,
    thresholds=thresholds,
    title='CDF of Code Token Lengths (BPE cl100k_base)',
    xlabel='Token Length (BPE tokens)'
)
plt.show()

## 5. Truncation Risk Quantification
Calculate the exact percentage of rows exceeding each context window threshold.

In [ ]:
print("="*80)
print("TRUNCATION RISK ANALYSIS")
print("="*80)
print()
print(f"{'Partition':<10} {'Total':<10} {'> 2048':<12} {'> 4096':<12} {'> 8192':<12}")
print("-"*56)

for space, lengths in token_lengths.items():
    total = len(lengths)
    gt_2048 = (lengths > 2048).sum()
    gt_4096 = (lengths > 4096).sum()
    gt_8192 = (lengths > 8192).sum()
    
    pct_2048 = 100 * gt_2048 / total
    pct_4096 = 100 * gt_4096 / total
    pct_8192 = 100 * gt_8192 / total
    
    print(f"{space:<10} {total:<10} {gt_2048} ({pct_2048:5.1f}%)   {gt_4096} ({pct_4096:5.1f}%)   {gt_8192} ({pct_8192:5.1f}%)")

print()
print("Interpretation:")
print("  Rows exceeding 2048 tokens will be TRUNCATED by small encoder models")
print("  like T5Gemma, potentially losing critical algorithmic information.")

## 6. Percentile Analysis
Detailed percentile breakdown per partition.

In [ ]:
percentiles = [10, 25, 50, 75, 90, 95, 99, 99.5]

print("Token Length Percentiles:")
print("="*80)

results = []
for space, lengths in token_lengths.items():
    row = {'partition': space}
    for p in percentiles:
        row[f'P{p}'] = int(np.percentile(lengths, p))
    results.append(row)

perc_df = pd.DataFrame(results).set_index('partition')
print(perc_df.to_string())

print("\n\ud83d\udcca Key observations:")
for space, lengths in token_lengths.items():
    median = np.median(lengths)
    p95 = np.percentile(lengths, 95)
    print(f"  {space}: median={median:.0f} tokens, P95={p95:.0f} tokens")

## 7. Token Length vs Target Correlation
Does code length correlate with performance? Longer code might be more complex.

In [ ]:
from scipy import stats as sp_stats

print("Token Length vs val_accuracy Correlation:")
print("="*60)

for space in ['APPS', 'CDSS', 'KBSS']:
    df = partitions[space]
    lengths = token_lengths[space]
    targets = df['val_accuracy'].values
    
    # Filter NaN
    valid = ~np.isnan(targets)
    if valid.sum() < 3:
        print(f"  {space}: insufficient valid data")
        continue
    
    rho, p = sp_stats.spearmanr(lengths[valid], targets[valid])
    print(f"  {space}: Spearman \u03c1 = {rho:+.4f} (p = {p:.2e})")

print("\n  Positive \u03c1 suggests longer code tends to use more resources,")
print("  confirming that truncation would remove potentially predictive information.")

## Conclusion

In [ ]:
print("="*80)
print("EXPERIMENT 4 CONCLUSION")
print("="*80)
print()
print("Context truncation analysis reveals:")
for space, lengths in token_lengths.items():
    pct = 100 * (lengths > 2048).sum() / len(lengths)
    print(f"  {space}: {pct:.1f}% of rows exceed 2048-token context window")
print()
print("If a significant portion exceeds 2048 tokens, the text-to-text")
print("regression paradigm will experience forced data attrition,")
print("validating the theoretical critique about structural truncation.")
if VALIDATION_MODE:
    print("\n\u26a0\ufe0f  Note: Percentages from validation sample. Run full mode for precise estimates.")